## Feature extraction functions

In [2]:
import pandas as pd

In [ ]:
def extract_deephase_score(seq, site):
    
    pass
    return score_df

In [7]:
def extract_idr_achor_score(seq, site):
    iupred_type = "long"
    # Predict disorder using iupred2a_lib
    iupred_scores = iupred2a_lib.iupred(sequence, iupred_type)[0]

    # Predict anchor regions using iupred2a_lib if anchor is enabled
    anchor_scores = iupred2a_lib.anchor2(sequence)[0]

    # Prepare the data for saving into a CSV file
    data = {
        "position": list(range(1, len(sequence) + 1)),
        "amino_acid": list(sequence),
        "iupred_score": iupred_scores,
        "anchor_score": anchor_scores,
    }

    # Create a DataFrame from the data
    df = pd.DataFrame(data)
    
    idr_score_df = df[df["position"] == site][["iupred_score", "anchor_score"]]
        
    return idr_score_df

In [1]:
#@title <b>Preliminary operations</b>
import subprocess
subprocess.run( 'pip install wget localcider==0.1.18'.split() )
subprocess.run('pip uninstall scikit-learn -y'.split())
subprocess.run('pip install scikit-learn==1.0.2'.split())
import numpy as np
import itertools
from localcider.sequenceParameters import SequenceParameters
import wget
import sys
import os
from joblib import dump, load
import pandas as pd
# from google.colab import files
from ipywidgets import IntProgress
from IPython.display import display
from IPython.display import clear_output

def calc_seq_prop(seq,residues,Nc,Cc,Hc):
    seq = list(seq).copy()
    fasta_kappa = np.array(seq.copy())
    N = len(seq)
    r = residues.copy()

    # calculate properties that do not depend on charges
    fK = sum([seq.count(a) for a in ['K']])/N
    fR = sum([seq.count(a) for a in ['R']])/N
    fE = sum([seq.count(a) for a in ['E']])/N
    fD = sum([seq.count(a) for a in ['D']])/N
    faro = sum([seq.count(a) for a in ['W','Y','F']])/N
    mean_lambda = np.mean(r.loc[seq].lambdas)
    
    pairs = np.array(list(itertools.combinations(seq,2)))
    pairs_indices = np.array(list(itertools.combinations(range(N),2)))
    # calculate sequence separations
    ij_dist = np.diff(pairs_indices,axis=1).flatten().astype(float)
    # calculate lambda sums
    ll = r.lambdas.loc[pairs[:,0]].values+r.lambdas.loc[pairs[:,1]].values
    # calculate SHD
    beta = -1
    shd = np.sum(ll*np.power(np.abs(ij_dist),beta))/N

    # fix charges
    if Nc == 1:
        r.loc['X'] = r.loc[seq[0]]
        r.loc['X','q'] = r.loc[seq[0],'q'] + 1.
        seq[0] = 'X'
        if r.loc['X','q'] > 0:
            fasta_kappa[0] = 'K'
        else:
            fasta_kappa[0] = 'A'
    if Cc == 1:
        r.loc['Z'] = r.loc[seq[-1]]
        r.loc['Z','q'] = r.loc[seq[-1],'q'] - 1.
        seq[-1] = 'Z'
        if r.loc['Z','q'] < 0:
            fasta_kappa[-1] = 'D'
        else:
            fasta_kappa[-1] = 'A'
    if Hc < 0.5:
        r.loc['H', 'q'] = 0
        fasta_kappa[np.where(np.array(seq) == 'H')[0]] = 'A'
    elif Hc >= 0.5:
        r.loc['H', 'q'] = 1
        fasta_kappa[np.where(np.array(seq) == 'H')[0]] = 'K'

    # calculate properties that depend on charges
    pairs = np.array(list(itertools.combinations(seq,2)))
    # calculate charge products
    qq = r.q.loc[pairs[:,0]].values*r.q.loc[pairs[:,1]].values
    # calculate SCD
    scd = np.sum(qq*np.sqrt(ij_dist))/N
    SeqOb = SequenceParameters(''.join(fasta_kappa))
    kappa = SeqOb.get_kappa()
    fcr = r.q.loc[seq].abs().mean()
    ncpr = r.q.loc[seq].mean()
    
    return pd.Series(data=[fK,fR,fE,fD,faro,scd,shd,kappa,fcr,mean_lambda,ncpr],
                 index=['fK','fR','fE','fD','faro','SCD','SHD','kappa','FCR','mean_lambda','NCPR'])

In [5]:
def extract_compactness_score(idr_seq):
    aa = ['A','C','D','E','F','G','H','I','K','L','M','N','P','Q','R','S','T','V','W','Y']

    url = 'https://github.com/KULL-Centre/_2023_Tesei_IDRome/blob/main'

    if os.path.exists('svr_model_nu.joblib') == False:
        wget.download(url+'/svr_models/svr_model_nu.joblib?raw=true')
    if os.path.exists('svr_model_SPR.joblib') == False:
        wget.download(url+'/svr_models/svr_model_SPR.joblib?raw=true')
    if os.path.exists('residues.csv') == False:
        wget.download(url+'/md_simulations/data/residues.csv?raw=true')

    model_nu = load('svr_model_nu.joblib') 
    model_spr = load('svr_model_SPR.joblib') 
    features_nu = ['SCD','SHD','kappa','FCR','mean_lambda']
    features_spr = ['SCD','SHD','mean_lambda']

    residues = pd.read_csv('residues.csv')
    residues = residues.set_index('one')

    fasta_dict = {}
    df = pd.DataFrame(columns=['nuSVR','SconfSVR/N (kB)','mean_lambda','SHD','SCD','kappa','FCR','NCPR',
                               'fK','fR','fE','fD','faro'])
    
    
    # Define the FASTA format: start with the header, then the sequence
    fasta_dict = {}
    protein_name = "protein1"  # You can specify any name, here I use "protein1"

    # Convert sequence string to FASTA format
    fasta_dict[protein_name] = idr_seq
    current_upload = [protein_name]

    # Validate the sequence
    for x in list(current_upload):  # Create a copy for safe modification
        valid = True
        for a in fasta_dict[x]:
            if a not in aa:
                print(f'WARNING: {x} sequence contains a character ({a}) not recognized as an amino acid. This sequence will be ignored.')
                del fasta_dict[x]
                valid = False
                break
        if not valid:
            current_upload.remove(x)

    # Output the results
    print("Valid sequences uploaded in FASTA format:")
    for name in fasta_dict:
        print(f">{name}")
        print(fasta_dict[name])
        
    #@title <b>Define charge states</b>
    #@markdown Define charge states:
    charged_N_terminal_amine = False #@param {type:"boolean"}
    charged_C_terminal_carboxyl = True #@param {type:"boolean"}
    charged_histidine = False #@param {type:"boolean"}
    Nc = 1 if charged_N_terminal_amine == True else 0
    Cc = 1 if charged_C_terminal_carboxyl == True else 0
    if charged_histidine == False:
        Hc = 0
        
    #@title <b>Predict $\nu$ and $S_\text{conf}/N$
    #@markdown Use this cell to calculate sequence features and predict the scaling exponent, $\nu$, and the conformational entropy per residue, $S_\text{conf}/N$. Results will be downloaded in a csv file.

    f = IntProgress(min=0, max=len(fasta_dict), description='Progress:', bar_style='warning')
    display(f)

    for k in fasta_dict.keys():
        res = calc_seq_prop(fasta_dict[k],residues,Nc,Cc,Hc)
        nu = np.around(model_nu.predict(res.loc[features_nu].values.reshape(1, -1))[0],3)
        spr = np.around(model_spr.predict(res.loc[features_spr].values.reshape(1, -1))[0],3)
        df.loc[k,'nuSVR'] = nu
        df.loc[k,'SconfSVR/N (kB)'] = spr
        df.loc[k,res.index.values] = np.around(res.loc[res.index.values].values,3)
        f.value += 1

    clear_output()
#     df.to_csv('svr_pred.csv',index_label='name')
    
    return df

In [8]:
idr_seq = "MSSQSHPDGLSGRDQPVELLNPARVNHMPSTVDVATALPLQVAPSAVPMDLRLDHQFSLPVAEPALREQQLQQELLALKQKQQIQRQILIAEFQRQHEQLSRQHEAQLHEHIKQQQEMLAMKHQQELLEHQRKLERHRQEQELEKQHREQKLQQLKNKEKGKESAVASTEVKMKLQEFVLNKKKALAHRNLNHCISSDPRYWYGKTQHSSLDQSSPPQSGVSTSYNHPVLGMYDAKDDFPLRKTASEPNLKLRSRLKQKVAERRSSPLLRRKDGPVVTALKKRPLDVTDSACSSAPGSGPSSPNNSSGSVSAENGIAPAVPSIPAETSLAHRLVAREGSAAPLPLYTSPSLPNITLGLPATGPSAGTAGQQDAERLTLPALQQRLSLFPGTHLTPYLSTSPLERDGGAAHSPLLQHMVLLEQPPAQAPLVTGLGALPLHAQSLVGADRVSPSIHKLRQHRPLGRTQSAPLPQNAQALQHLVIQQQHQQFLEKHKQQFQQQQLQMNKIIPKPSEPARQPESHPEETEEELREHQALLDEPYLDRLPGQKEAHAQAGVQVKQEPIESDEEEAEPPREVEPGQRQPSEQELLFRQQALLLEQQRIHQLRNYQASMEAAGIPVSFGGHRPLSRAQSSPASATFPVSVQEPPTKPRFTTGLVYDTLMLKHQCTCGSSSSHPEHAGRIQSIWSRLQETGLRGKCECIRGRKATLEELQTVHSEAHTLLYGTNPLNRQKLDSKKLLGSLASVFVRLPCGGVGVDSDTIWNEVHSAGAARLAVGCVVELVFKVATGELKNGFAVVRPPGHHAEESTPMGFCYFNSVAVAAKLLQQRLSVSKILIVDWDVHHGNGTQQAFYSDPSVLYMSLHRYDDGNFFPGSGAPDEVGTGPGVGFNVNMAFTGGLDPPMGDAEYLAAFRTVVMPIASEFAPDVVLVSSGFDAVEGHPTPLGGYNLSARCFGYLTKQLMGLAGGRIVLALEGGHDLTAICDASEACVSALLGNELDPLPEKVLQQRPNANAVRSMEKVMEIHSKYWRCLQRTTSTAGRSLIEAQTCENEEAETVTAMASLSVGVKPAEKRPDEEPMEEEPPL"
extract_compactness_score(idr_seq)

,nuSVR,SconfSVR/N (kB),mean_lambda,SHD,SCD,kappa,FCR,NCPR,fK,fR,fE,fD,faro
protein1,0.488,9.911,0.425,5.598,-2.745,0.192,0.214,-0.011,0.045,0.056,0.077,0.034,0.044


In [ ]:
# def extract_disoflag_score(seq, site):
#     pass
    
#     return score_df

In [ ]:
def extract_phospholingo_score(seq, site):
    pass
    return score_df

In [ ]:
def extract_esm_embedding(seq, site):
    pass
    return protein_embedding_df, site_embedding_df

In [ ]:
def extract_ptmmamba_embedding(seq, site):
    pass
    return protein_embedding_df, site_embedding_df

In [22]:
def one_hot_encode(sequence):
    """
    One-hot encodes a protein sequence.

    Parameters:
        sequence (str): The protein sequence to be one-hot encoded.

    Returns:
        np.ndarray: A one-hot encoded representation of the sequence.
    """
    # Define the set of amino acids, including the "-" character
    amino_acids = 'ACDEFGHIKLMNPQRSTVWY-'
    one_hot = np.zeros((len(sequence), len(amino_acids)), dtype=int)

    for i, char in enumerate(sequence):
        if char in amino_acids:
            index = amino_acids.index(char)
            one_hot[i, index] = 1
        else:
            raise ValueError(f"Unexpected character '{char}' in sequence.")

    return one_hot, amino_acids

def extract_onehot_embedding(sequence, site):
    """
    Extracts subsequences around specified phosphorylation sites and one-hot encodes them.

    Parameters:
        seq (str): The name of the column containing sequences.
        site (int): The name of the column containing phosphorylation site (not site index).

    Returns:
        pd.DataFrame: The original DataFrame with additional one-hot encoded columns.
    """
    amino_acids = 'ACDEFGHIKLMNPQRSTVWY-'
    data = {}  # Dictionary to hold data for new columns

    # Initialize a sequence of 15 "-" characters
    sub_sequence = ['-'] * 15

    # change site into sequence index
    site -= 1
    
    # Determine the start and end indices for the window around the phosphorylation site
    start = max(0, site - 7)
    end = min(len(sequence), site + 8)  # +8 to include the site itself

    # Fill the sub_sequence list with the valid portion of the sequence
    for i in range(start, end):
        sub_sequence[i - (site - 7)] = sequence[i]

    # One-hot encode the extracted sequence
    one_hot, _ = one_hot_encode(''.join(sub_sequence))

    # Create new entries for the DataFrame
    for pos in range(-7, 8):  # From -7 to +7 (15 positions)
        for aa_index, aa in enumerate(amino_acids):
            column_name = f"onehot_{pos}_{aa}"
            # Initialize the list if the column doesn't exist in the data dictionary
            if column_name not in data:
                data[column_name] = []  # Initialize with an empty list

            # Append the one-hot encoded value to the list
            data[column_name].append(one_hot[pos + 7, aa_index])


    # Create a new DataFrame for the new columns with the same index as the original DataFrame
    new_df = pd.DataFrame(data)

    # Concatenate the new DataFrame with the original one
    df = pd.concat([new_df], axis=1)

    print("Successfully preprocess the file.")

    return df, df.shape

In [31]:
# Example usage
sequence = "AAAAAAA-AAAAAAAAAAA"
site = 1

region_sequence = extract_onehot_embedding(sequence, site)
region_sequence[0]

Successfully preprocess the file.


,onehot_-7_A,onehot_-7_C,onehot_-7_D,onehot_-7_E,onehot_-7_F,onehot_-7_G,onehot_-7_H,onehot_-7_I,onehot_-7_K,onehot_-7_L,...,onehot_7_N,onehot_7_P,onehot_7_Q,onehot_7_R,onehot_7_S,onehot_7_T,onehot_7_V,onehot_7_W,onehot_7_Y,onehot_7_-
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1


In [33]:
import os
import sys
import pandas as pd
import numpy as np
import src.files.iupred2a.iupred2a_lib as iupred2a_lib
from Bio import SeqIO
def extract_idr_seq(sequence, site):
    iupred_type = "long"
    # Predict disorder using iupred2a_lib
    iupred_scores = iupred2a_lib.iupred(sequence, iupred_type)[0]

    # Predict anchor regions using iupred2a_lib if anchor is enabled
    anchor_scores = iupred2a_lib.anchor2(sequence)[0]

    # Prepare the data for saving into a CSV file
    data = {
        "position": list(range(1, len(sequence) + 1)),
        "amino_acid": list(sequence),
        "iupred_score": iupred_scores,
        "anchor_score": anchor_scores,
    }

    # Create a DataFrame from the data
    df = pd.DataFrame(data)
    
    # Step 1: Add the 'call_idr' column
    df['call_idr'] = df['iupred_score'].apply(lambda x: 1 if x > 0.5 else 0)
    
    # Check if the provided site is within a '1' region
    if df.at[site-1, 'call_idr'] != 1:
        return None

    # Find the contiguous segments where call_idr == 1
    df['segment_id'] = (df['call_idr'] != df['call_idr'].shift()).cumsum()
    contiguous_segments = df[df['call_idr'] == 1].groupby('segment_id')
    
    for _, segment in contiguous_segments:
        # Check if the site falls within this segment
        if site in segment['position'].values:
            # Truncate the segment to a maximum length of 500 if necessary
            start_idx = segment.index[0]
            end_idx = segment.index[-1]

            if len(segment) > 500:
                # Ensure the site is within the truncated segment
                site_idx = df.index[df['position'] == site][0]
                if site_idx - start_idx > 250:
                    start_idx = max(site_idx - 250, start_idx)
                end_idx = min(start_idx + 500, end_idx)

            truncated_segment = df.loc[start_idx:end_idx]
            return "".join(truncated_segment['amino_acid'].tolist())

    return None

In [34]:
# Example usage
sequence = "MSSQSHPDGLSGRDQPVELLNPARVNHMPSTVDVATALPLQVAPSAVPMDLRLDHQFSLPVAEPALREQQLQQELLALKQKQQIQRQILIAEFQRQHEQLSRQHEAQLHEHIKQQQEMLAMKHQQELLEHQRKLERHRQEQELEKQHREQKLQQLKNKEKGKESAVASTEVKMKLQEFVLNKKKALAHRNLNHCISSDPRYWYGKTQHSSLDQSSPPQSGVSTSYNHPVLGMYDAKDDFPLRKTASEPNLKLRSRLKQKVAERRSSPLLRRKDGPVVTALKKRPLDVTDSACSSAPGSGPSSPNNSSGSVSAENGIAPAVPSIPAETSLAHRLVAREGSAAPLPLYTSPSLPNITLGLPATGPSAGTAGQQDAERLTLPALQQRLSLFPGTHLTPYLSTSPLERDGGAAHSPLLQHMVLLEQPPAQAPLVTGLGALPLHAQSLVGADRVSPSIHKLRQHRPLGRTQSAPLPQNAQALQHLVIQQQHQQFLEKHKQQFQQQQLQMNKIIPKPSEPARQPESHPEETEEELREHQALLDEPYLDRLPGQKEAHAQAGVQVKQEPIESDEEEAEPPREVEPGQRQPSEQELLFRQQALLLEQQRIHQLRNYQASMEAAGIPVSFGGHRPLSRAQSSPASATFPVSVQEPPTKPRFTTGLVYDTLMLKHQCTCGSSSSHPEHAGRIQSIWSRLQETGLRGKCECIRGRKATLEELQTVHSEAHTLLYGTNPLNRQKLDSKKLLGSLASVFVRLPCGGVGVDSDTIWNEVHSAGAARLAVGCVVELVFKVATGELKNGFAVVRPPGHHAEESTPMGFCYFNSVAVAAKLLQQRLSVSKILIVDWDVHHGNGTQQAFYSDPSVLYMSLHRYDDGNFFPGSGAPDEVGTGPGVGFNVNMAFTGGLDPPMGDAEYLAAFRTVVMPIASEFAPDVVLVSSGFDAVEGHPTPLGGYNLSARCFGYLTKQLMGLAGGRIVLALEGGHDLTAICDASEACVSALLGNELDPLPEKVLQQRPNANAVRSMEKVMEIHSKYWRCLQRTTSTAGRSLIEAQTCENEEAETVTAMASLSVGVKPAEKRPDEEPMEEEPPL"
site = 246

region_sequence = extract_idr_seq(sequence, site)
region_sequence

'MYDAKDDFPLRKTASEPNLKLRSRLKQKVAERRSSPLLRRKDGPVVTALKKRP'

In [30]:
df

,position,amino_acid,iupred_score,anchor_score,call_idr,segment_id
0,1,M,0.891961,0.873385,1,1
1,2,S,0.901269,0.873385,1,1
2,3,S,0.901269,0.873385,1,1
3,4,Q,0.921076,0.873385,1,1
4,5,S,0.921076,0.873385,1,1
...,...,...,...,...,...,...
1079,1080,E,0.967676,0.873385,1,97
1080,1081,E,0.963420,0.873385,1,97
1081,1082,P,0.971713,0.873385,1,97
1082,1083,P,0.978316,0.873385,1,97


In [22]:
contiguous_segments

In [ ]:
deephase_score_df = extract_deephase_score(seq, site)
idr_score_df = extract_idr_achor_score(seq, site)
disoflag_score_df = extract_disoflag_score(seq, site)
phospholingo_score_df = extract_phospholingo_score(seq, site)

In [ ]:
protein_embedding_df, site_embedding_df = extract_esm_embedding(seq, site)
phosphoprotein_embedding_df, phosphosite_embedding_df = extract_ptmmamba_embedding(seq, site)
onehot_embedding_df = extract_onehot_embedding(seq, site, left_flank, right_flank)

In [ ]:
compactness_score_df = extract_compactness_score(idr_seq, site)

In [ ]:
# merge feature dfs
plm_features_df = pd.concat([protein_embedding_df, site_embedding_df, phosphoprotein_embedding_df, phosphosite_embedding_df], axis=1)
bio_features_df = pd.concat([deephase_score_df, idr_score_df, achor_score_df, disoflag_score_df, phospholingo_score_df, onehot_embedding_df], axis=1)

In [ ]:
def mutate_seq(seq, *mutation, site):
    pass
    return mutated_seq

In [ ]:
def build_mutation_lst(mutation_string):
    pass
    return site, original_aa, mutated_aa

In [3]:
# build the features df:
curated_dataset_df = pd.read_csv("Table 1.csv")
curated_dataset_df = curated_dataset_df[curated_dataset_df["label"]==1]
curated_dataset_df

,human homology accession,human homology site without letter,human Site and Mutation,label,PMID
0,P43681,467,S467s,1,20141511
1,P11362,779,S779s,1,23564461
2,P98177,32,T32t,1,20141511
3,P14136,8,S8s,1,20141511
4,P08151,640,S640s,1,20141511
...,...,...,...,...,...
781,Q53ET0,368,S368s,1,18626018
782,Q05086-2,485,T485t,1,28835500
783,P40818,718,S718s/S716A,1,17720156
784,P40818,718,S718s,1,17720156


In [ ]:
# build the features df:
def build_features_df(uniprot, site, mutation_string):
    wt_seq = fetch_seq(uniprot)
    mut_lst = build_mutation_lst(mutation_string)
    if mut_lst:
        seq = mutate_seq(wt_seq, *mut_lst, site)
    else:
        seq = wt_seq
        
    deephase_score_df = extract_deephase_score(seq, site)
    idr_score_df, achor_score_df = extract_idr_achor_score(seq, site)
    disoflag_score_df = extract_disoflag_score(seq, site)
    phospholingo_score_df = extract_phospholingo_score(seq, site)
    protein_embedding_df, site_embedding_df = extract_esm_embedding(seq, site)
    phosphoprotein_embedding_df, phosphosite_embedding_df = extract_ptmmamba_embedding(seq, site)
    onehot_embedding_df = extract_onehot_embedding(seq, site)
    
    idr_seq = extract_idr_seq(seq, site)
    compactness_score_df = extract_compactness_score(idr_seq, site)
    
    # merge feature dfs
    plm_features_df = pd.concat([protein_embedding_df, site_embedding_df, phosphoprotein_embedding_df, phosphosite_embedding_df], axis=1)
    bio_features_df = pd.concat([deephase_score_df, idr_score_df, achor_score_df, disoflag_score_df, phospholingo_score_df, onehot_embedding_df], axis=1)
    
    return plm_features_df,bio_features_df


## Training dataset, independent dataset split

### 4.3	Feature selection 

task 36: modeling direct from only PLM embedding, we first compare using embeddings from either ESM2 or PTM-mamba to represent the whole protein or the residue embeddings, 

task 37: Then we tested modeling from multi-scale biology features plus onehot encoding for the -7 to +7 motif, the result show that multi-scale biology features also can be used to predict the 14-3-3 binding when compare with only use onehot encoded motif feature. 

task 38: We then test using both PLM and biology features, the results show that using both PLM and biology features shows improved results compared with using only PLM or multi-scale biology features. 

task 39: feature selection using mRMR. We mainly test how many features are optimized for our model by using different number of features to fit the model and compare the model performance.

task 40: We also test feature selection using another method Boruta, it shows that XX feature give us the best results.

task 41: feature importance and its contribution were
further analyzed to find which feature was more valuable for
the model performance after feature selection 

### 4.4	Prediction performance with different classifiers

task 42: To test the validity of the optimal feature set in different classifiers, three common classifiers were used to predict 14-3-3 sites: random forest (RF), XGBoost (XGB), SVM and ANN are used. 

## 4.5	Model architecture and optimization 

task 43: fine-tuned certain hyperparameters (Table X) of the model

## 4.6 Feature Importance analysis

task 44: we conducted a thorough examination of feature importance using SHAP (SHapley Additive exPlanations) values

task 45: ablation experiments where we systematically removed features to observe changes in model performance.

## 4.7	Performance evaluation and comparison with existing methods, Validation on independent test sets

task 46: Independent datset test with 14-3-3 pred and 14-3-3 site-finder, compare with our results.

task 47: download and curate prediction data from clinvar

task 48: prediction

task 49: figure

task: model update